In [1]:
# model training

import os
import pickle
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier, XGBRegressor

# -----------------------------------------
# Paths
# -----------------------------------------
INPUT = "df_enhanced_fixed.csv"
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Load data
df = pd.read_csv(INPUT, parse_dates=["Date"], dayfirst=False)

# Sort inside company
df = df.sort_values(["Company_Cleaned", "Date"], kind="mergesort")

# Create labels (next round)
df["next_date"] = df.groupby("Company_Cleaned")["Date"].shift(-1)
df["next_amount"] = df.groupby("Company_Cleaned")["Amount_Cr"].shift(-1)

df["time_to_next_round"] = (df["next_date"] - df["Date"]).dt.days

# Classification target
df["raised_again_12m"] = np.where(
    df["time_to_next_round"].notna() & (df["time_to_next_round"] <= 365), 
    1,
    np.where(df["time_to_next_round"].notna(), 0, np.nan)
)

# Regression target
df["amount_next_round"] = df["next_amount"]

# Feature list
features = [
    "Amount_Cr",
    "Cumulative_Funding_Prior",
    "Rolling_6m_Funding_Lagged",
    "Rolling_12m_Funding_Lagged",
    "Rolling_6m_Rounds_Lagged",
    "Rolling_12m_Rounds_Lagged",
    "Sector",
    "Sub_Sector",
    "city",
    "state",
]

df_feat = df.dropna(subset=features)

# Categorical + Numeric split
cat_cols = ["Sector", "Sub_Sector", "city", "state"]
num_cols = [c for c in features if c not in cat_cols]

preprocess = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

# =====================================================
# MODEL 1: WILL RAISE AGAIN? (CLASSIFIER)
# =====================================================
df_cls = df_feat[df_feat["raised_again_12m"].notna()].copy()

X1 = df_cls[features]
y1 = df_cls["raised_again_12m"].astype(int)

clf_pipeline = Pipeline([
    ("prep", preprocess),
    ("clf", XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=-1
    )),
])

clf_pipeline.fit(X1, y1)

pickle.dump(
    clf_pipeline,
    open(os.path.join(MODEL_DIR, "funding_round_model.pkl"), "wb")
)

print("Saved: funding_round_model.pkl")

# =====================================================
# MODEL 2: PREDICT NEXT AMOUNT (REGRESSOR)
# =====================================================
df_reg = df_feat[df_feat["amount_next_round"].notna()].copy()

X2 = df_reg[features]
y2 = df_reg["amount_next_round"].astype(float)

reg_pipeline = Pipeline([
    ("prep", preprocess),
    ("reg", XGBRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=6,
        subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=-1
    )),
])

reg_pipeline.fit(X2, y2)

pickle.dump(
    reg_pipeline,
    open(os.path.join(MODEL_DIR, "funding_amount_model.pkl"), "wb")
)

print("Saved: funding_amount_model.pkl")

# =====================================================
# MODEL 3: INVESTMENT SCORE (CLASSIFIER)
# =====================================================
score_pipeline = Pipeline([
    ("prep", preprocess),
    ("clf", XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=-1
    )),
])

score_pipeline.fit(X1, y1)

pickle.dump(
    score_pipeline,
    open(os.path.join(MODEL_DIR, "invest_score_model.pkl"), "wb")
)

print("Saved: invest_score_model.pkl")
print("ALL MODELS TRAINED SUCCESSFULLY")


Saved: funding_round_model.pkl
Saved: funding_amount_model.pkl
Saved: invest_score_model.pkl
ALL MODELS TRAINED SUCCESSFULLY


In [4]:
# model evaluation 

import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, r2_score, mean_squared_error
)

# -------------------------------
# Load full enhanced dataset
# -------------------------------
df = pd.read_csv("df_enhanced_fixed.csv", parse_dates=["Date"])

# -------------------------------
# FEATURES USED BY MODELS
# -------------------------------
features = [
    "Amount_Cr",
    "Cumulative_Funding_Prior",
    "Rolling_6m_Funding_Lagged",
    "Rolling_12m_Funding_Lagged",
    "Rolling_6m_Rounds_Lagged",
    "Rolling_12m_Rounds_Lagged",
    "Sector",
    "Sub_Sector",
    "city",
    "state",
]

# -------------------------------
# RECREATE LABELS (same as training)
# -------------------------------
df = df.sort_values(["Company_Cleaned", "Date"], kind="mergesort")

# Next funding date and amount
df["next_date"] = df.groupby("Company_Cleaned")["Date"].shift(-1)
df["next_amount"] = df.groupby("Company_Cleaned")["Amount_Cr"].shift(-1)

df["time_to_next_round"] = (df["next_date"] - df["Date"]).dt.days

df["raised_again_12m"] = np.where(
    df["time_to_next_round"].notna() & (df["time_to_next_round"] <= 365), 1,
    np.where(df["time_to_next_round"].notna(), 0, np.nan)
)

# -------------------------------
# LOAD MODELS
# -------------------------------
fund_model = pickle.load(open("models/funding_round_model.pkl", "rb"))
amount_model = pickle.load(open("models/funding_amount_model.pkl", "rb"))
score_model = pickle.load(open("models/invest_score_model.pkl", "rb"))

# ============================================================
# 1. CLASSIFICATION MODEL EVALUATION (Will Raise Again)
# ============================================================
df_cls = df.dropna(subset=features + ["raised_again_12m"])

X_cls = df_cls[features]
y_cls = df_cls["raised_again_12m"].astype(int)

pred_prob = fund_model.predict_proba(X_cls)[:, 1]
pred_label = (pred_prob >= 0.5).astype(int)

print("\n===== MODEL 1 — WILL RAISE AGAIN =====")
print("Accuracy :", accuracy_score(y_cls, pred_label))
print("Precision:", precision_score(y_cls, pred_label))
print("Recall   :", recall_score(y_cls, pred_label))
print("F1 Score :", f1_score(y_cls, pred_label))
print("AUC      :", roc_auc_score(y_cls, pred_prob))

# ============================================================
# 2. REGRESSION MODEL EVALUATION (Next Amount)
# ============================================================
df_reg = df.dropna(subset=features + ["next_amount"])

X_reg = df_reg[features]
y_reg = df_reg["next_amount"].astype(float)

pred_reg = amount_model.predict(X_reg)

print("\n===== MODEL 2 — NEXT ROUND AMOUNT =====")
print("R²   :", r2_score(y_reg, pred_reg))
print("RMSE :", np.sqrt(mean_squared_error(y_reg, pred_reg)))

# ============================================================
# 3. INVESTMENT SUCCESS SCORE MODEL
# ============================================================
pred_score_prob = score_model.predict_proba(X_cls)[:, 1]

print("\n===== MODEL 3 — INVESTOR SUCCESS SCORE =====")
print("AUC :", roc_auc_score(y_cls, pred_score_prob))

print("\n🎉 All evaluation metrics computed successfully!")



===== MODEL 1 — WILL RAISE AGAIN =====
Accuracy : 0.9064919594997022
Precision: 0.9096989966555183
Recall   : 0.9147982062780269
F1 Score : 0.9122414756847401
AUC      : 0.9734018609580573

===== MODEL 2 — NEXT ROUND AMOUNT =====
R²   : 0.95889503617984
RMSE : 207.99752758407075

===== MODEL 3 — INVESTOR SUCCESS SCORE =====
AUC : 0.9734018609580573

🎉 All evaluation metrics computed successfully!
